# Notebook 02: Cleaning and Transformation
This notebook applies data cleaning rules, builds processed CSV outputs, and validates the transformed data ready for analytics.

In [ ]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd()
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

nav_history = pd.read_csv(RAW_DIR / "02_nav_history.csv")
performance = pd.read_csv(RAW_DIR / "07_scheme_performance.csv")
transactions = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")

print(nav_history.shape)
print(performance.shape)
print(transactions.shape)

In [ ]:
# Clean NAV history
nav_history = nav_history[["amfi_code", "date", "nav"]].copy()
nav_history["amfi_code"] = pd.to_numeric(nav_history["amfi_code"], errors="coerce")
nav_history["date"] = pd.to_datetime(nav_history["date"], errors="coerce")
nav_history["nav"] = pd.to_numeric(nav_history["nav"], errors="coerce")
nav_history = nav_history.dropna(subset=["amfi_code", "date", "nav"]).sort_values(["amfi_code", "date"])
nav_history = nav_history.drop_duplicates(subset=["amfi_code", "date"])
nav_history.to_csv(PROCESSED_DIR / "nav_history_clean.csv", index=False)
nav_history.head()

In [ ]:
# Clean investor transactions
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], errors="coerce")
transactions["amount"] = pd.to_numeric(transactions["amount_inr"], errors="coerce")
transactions = transactions.dropna(subset=["transaction_date", "amount"])
transactions["transaction_type"] = transactions["transaction_type"].fillna("Lumpsum")
transactions["transaction_type"] = transactions["transaction_type"].replace(
    {"sip": "SIP", "lumpsum": "Lumpsum", "redemption": "Redemption"}
)
transactions_clean = transactions[["investor_id", "transaction_date", "amfi_code", "transaction_type", "amount", "state", "kyc_status"]].copy()
transactions_clean.to_csv(PROCESSED_DIR / "investor_transactions_clean.csv", index=False)
transactions_clean.head()

In [ ]:
# Clean performance data
performance = performance.rename(columns={
    "return_1yr_pct": "return_1y",
    "return_3yr_pct": "return_3y",
    "return_5yr_pct": "return_5y",
    "expense_ratio_pct": "expense_ratio",
})
for col in ["return_1y", "return_3y", "return_5y", "expense_ratio"]:
    performance[col] = pd.to_numeric(performance[col], errors="coerce")
performance = performance.dropna(subset=["amfi_code"])
performance.to_csv(PROCESSED_DIR / "scheme_performance_clean.csv", index=False)
performance.head()

## Summary
- Cleaned NAV history, investor transactions, and performance snapshots.
- Saved standardized processed CSV files under `data/processed`.
- Next, we will build the analytics tables and verify the normalized SQLite schema.